In [64]:
import serial
import serial.tools.list_ports
import sys
import time
import ctypes
import csv
import os
import datetime

### PKT SETUP

In [ ]:
# Setup configurations here:

baudrate = 115200 # serial baud



# ---------- PKT STRUCTURE ----------

# Bytes for each field
SYNC_WORD_BYTES  = 0
MASK_BYTES       = 1
NODE_ID_BYTES    = 1
BC_ID_BYTES      = 1
PKT_ID_BYTES     = 2
RSSI_BYTES       = 2
SENSOR_PLD_BYTES = 36

# Position (IDX) in the RX Buffer
SYNC_WORD_POS  = 0

MASK_POS        = SYNC_WORD_POS + SYNC_WORD_BYTES
ALARM_BIT_POS   = 0
ACK_BIT_POS     = 1
RETX_BIT_POS    = 2

MASK_ALARM_BIT  = (1 << ALARM_BIT_POS)
MASK_ALARM_ACK  = (1 << ACK_BIT_POS)
MASK_RETX_BIT   = (1 << RETX_BIT_POS)

NODE_ID_POS    = MASK_POS + MASK_BYTES
PKT_ID_MSB_POS = NODE_ID_POS + NODE_ID_BYTES
PKT_ID_LSB_POS = NODE_ID_POS + NODE_ID_BYTES + 1
SMPL_DATA_POS  = PKT_ID_MSB_POS + PKT_ID_BYTES
RSSI_POS       = SMPL_DATA_POS + SENSOR_PLD_BYTES
BC_ID1_POS     = RSSI_POS + RSSI_BYTES


# -------- Config protocol  --------
SYNC_WORD_ENV = b'\xAA\x55'          
ENV_NODE_PYL_SIZE = 40          # removed sync bytes wrt stm32 code      

SYNC_WORD_BC = b'\x11\xAA'           
BC_NODE_MIN_PYL_SIZE = 43             


### Public functions

In [59]:

# USE THIS FUNC TO SELECT PORT
def select_port():
    ports = serial.tools.list_ports.comports()
    if not ports:
        print("Nessuna porta seriale trovata!")
        return None
    
    print("Porte disponibili:")
    for i, port in enumerate(ports):
        print(f"{i}: {port.device} ({port.description})")
    
    selection = input("Seleziona il numero della porta: ")
    
    try:
        idx = int(selection)
        return ports[idx].device
    except (ValueError, IndexError):
        print("Selezione non valida!")
        return None


# Use this function to listen to the serial port
# and parse pkts
def readSerial(my_port, csv_writer, file_log):
    try:
        # Close if open from prev run
        if 'ser' in locals() and ser.is_open:
            ser.close()
        
        # Open
        ser = serial.Serial(my_port, baudrate, timeout=0.1)
        print(f"--- Listening {my_port} ({baudrate} bps) ---")

        while True:
            # look for sync char
            first_byte = ser.read(1)
            if not first_byte: continue

            # if ENVIRONMENTAL NODE
            if first_byte == SYNC_WORD_ENV[0:1]:
                second_byte = ser.read(1)
                if second_byte == SYNC_WORD_ENV[1:2]:
                    # This is an ENV NODE
                    payload = ser.read(ENV_NODE_PYL_SIZE)

                    # Parse payload:
                    mask   = payload[MASK_POS] 
                    nodeID = payload[NODE_ID_POS]
                    pktID  = (payload[PKT_ID_MSB_POS] << 8) | (payload[PKT_ID_LSB_POS])

                    smpl_payload = payload[SMPL_DATA_POS:RSSI_POS]

                    print("\n\nNew ENV NODE PKT")
                    print("MASK:",mask)
                    print("NODE_ID:",nodeID)
                    print("PKT_ID:",pktID)
                    print("Payload:",smpl_payload)


                    # --- Save on file ---
                    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    # Empty slots to maintain alignment between ENV node and BC node (some fields not present in ENV nodes pkts)
                    csv_writer.writerow([timestamp, 'ENV', mask,nodeID,pktID,smpl_payload.hex(),'', ''])
                    f_log.flush() # Force writing now

                    continue

            # if BC_NODE
            if first_byte == SYNC_WORD_BC[0:1]:
                second_byte = ser.read(1)
                if second_byte == SYNC_WORD_BC[1:2]:
                    # This is a BC NODE

                    # Read up to payload bytes (exclude BC sequence)
                    payload_bytes = ser.read(BC_ID1_POS)

                    # Parse payload:
                    mask   = payload_bytes[MASK_POS] 
                    alrm_bit     = mask & MASK_ALARM_BIT
                    alrm_ack_bit = mask & MASK_ALARM_ACK
                    
                    pkt_retx     = mask & MASK_RETX_BIT
                    
                    nodeID = payload_bytes[NODE_ID_POS]
                    pktID  = (payload_bytes[PKT_ID_MSB_POS] << 8) | (payload_bytes[PKT_ID_LSB_POS])

                    print("\n\nNew BC NODE PKT")
                    print("MASK:",mask, "(ALARM:",alrm_bit,"ALARM ACK:",alrm_ack_bit,")")
                    print("NODE_ID:",nodeID)
                    print("PKT_ID:",pktID)

                    smpl_payload = payload_bytes[SMPL_DATA_POS:RSSI_POS]
                    print("Payload:",smpl_payload)

                    rssi_raw = (payload_bytes[RSSI_POS] << 8) | payload_bytes[RSSI_POS+1]
                    rssi = ctypes.c_int16(rssi_raw).value

                    print("RSSI:",rssi)
                    
                    # Read BCs sequence:
                    bcs_num = ser.read(1)[0]
                    bcs_seq = ser.read(bcs_num)

                    print("BC sequence:", bcs_seq)

                    # --- Save on file ---
                    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    # Empty slots to maintain alignment between ENV node and BC node (some fields not present in ENV nodes pkts)
                    csv_writer.writerow([timestamp,'BC',mask,nodeID,pktID,smpl_payload.hex(),rssi,bcs_seq.hex()])
                    f_log.flush() # Force writing now

                    continue

            # if ser.in_waiting > 0:
            #     data = ser.read(ser.in_waiting)
                
            #     hex_data = " ".join(f"{b:02X}" for b in data)
            #     print(f"RX {len(data)} byte: {hex_data}")
            
            time.sleep(0.01) 

    except Exception as e:
        print(f"\nError: {e}")

    finally:
        # in the end ensure serial is closed
        if 'ser' in locals() and ser.is_open:
            ser.close()
            print("\n--- Port closed ---")

### Select serial Port

In [55]:
my_port = select_port()

Porte disponibili:
0: /dev/cu.wlan-debug (n/a)
1: /dev/cu.debug-console (n/a)
2: /dev/cu.Bluetooth-Incoming-Port (n/a)
3: /dev/cu.usbmodem1102 (STLINK-V3)


### Launch application

In [68]:
# --- SETUP FILE CSV ---
filename = "serial_log.csv"
file_exists = os.path.isfile(filename)

# Apriamo il file in modalità append ('a') prima del try
f_log = open(filename, 'a', newline='')
csv_writer = csv.writer(f_log)

# Write file header only if new
if not file_exists:
    csv_writer.writerow(['Timestamp', 'Type', 'Mask', 'NodeID', 'PktID', 'Payload', 'RSSI', 'BC_Seq'])
    f_log.flush()

readSerial(my_port, csv_writer, f_log)

--- Listening /dev/cu.usbmodem1102 (115200 bps) ---


New BC NODE PKT
MASK: 1 (ALARM: 1 ALARM ACK: 0 )
NODE_ID: 1
PKT_ID: 1
Payload: b'\xaa\xaa\xbb\xbb\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
RSSI: -92
BC sequence: b'\x05\x04\x03'


New BC NODE PKT
MASK: 1 (ALARM: 1 ALARM ACK: 0 )
NODE_ID: 1
PKT_ID: 1
Payload: b'\xaa\xaa\xbb\xbb\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
RSSI: -89
BC sequence: b'\x05\x04\x01'


New BC NODE PKT
MASK: 3 (ALARM: 1 ALARM ACK: 2 )
NODE_ID: 1
PKT_ID: 1
Payload: b'\xaa\xaa\xbb\xbb\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
RSSI: -92
BC sequence: b'\x05\x04\x03'

--- Port closed ---


KeyboardInterrupt: 